# CAS Exam 5: Frequency-Severity Techniques and Disposal Rate Method

This notebook demonstrates:
1. Frequency-Severity Technique #1
2. Frequency-Severity Technique #2
3. Disposal Rate Method

Data used from `chainladder/utils/data`:
- `friedland_xyz_freq_sev.csv`
- `friedland_xyz_disp.csv`
- `xyz.csv` (premium used as an exposure proxy for Technique #2)


## Method Summaries (Formula-Sheet Aligned)

Frequency-Severity Technique #1:
- Project ultimate claim counts.
- Project ultimate severity.
- Ultimate claims = Ultimate counts x Ultimate severity.

Frequency-Severity Technique #2:
- Project ultimate claim counts.
- Convert to frequency using exposure and select frequency with trend consideration.
- Select severity with trend consideration.
- Ultimate claims = Exposure x Selected frequency x Selected severity.

Disposal Rate Method:
- Disposal rate at maturity t: DR_t = Cumulative closed counts_t / Ultimate closed counts.
- Rearranged: Ultimate closed counts = Cumulative closed counts_t / DR_t.
- Use selected disposal rates + selected incremental severities to project future unpaid.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import chainladder as cl

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from reservingengine.reserving import triangle_to_frame

DATA_DIR = ROOT / 'chainladder-python' / 'chainladder' / 'utils' / 'data'
freq_sev_df = pd.read_csv(DATA_DIR / 'friedland_xyz_freq_sev.csv')
disp_df = pd.read_csv(DATA_DIR / 'friedland_xyz_disp.csv')
xyz_df = pd.read_csv(DATA_DIR / 'xyz.csv')

freq_sev_triangle = cl.Triangle(
    freq_sev_df,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Closed Claim Counts', 'Reported Claim Counts', 'Reported Claims', 'Reported Severities'],
    cumulative=True,
)

closed_count_triangle = freq_sev_triangle['Closed Claim Counts']
reported_claims_triangle = freq_sev_triangle['Reported Claims']
reported_severity_triangle = freq_sev_triangle['Reported Severities']

{'freq_sev_triangle_shape': freq_sev_triangle.shape, 'valuation_date': str(freq_sev_triangle.valuation_date)}


In [ ]:
latest_counts = closed_count_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_claims = reported_claims_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_severity = reported_severity_triangle.latest_diagonal.to_frame().iloc[:, 0]

closed_long = triangle_to_frame(closed_count_triangle, origin_as_datetime=False).reset_index()
closed_matrix = closed_long.pivot(index='origin', columns='development', values='Closed Claim Counts').sort_index().sort_index(axis=1)
latest_age = closed_matrix.notna().iloc[:, ::-1].idxmax(axis=1)
latest_age.index = pd.to_datetime(latest_age.index.astype(str) + "-01-01")
latest_age = latest_age.reindex(latest_counts.index)

triangle_snapshot = pd.DataFrame(
    {
        'LatestClosedCounts': latest_counts.values,
        'LatestReportedClaims': latest_claims.values,
        'LatestReportedSeverity': latest_severity.values,
        'LatestMaturityAge': latest_age.values,
    },
    index=latest_counts.index.year,
)
triangle_snapshot.index.name = 'AccidentYear'
triangle_snapshot


## Frequency-Severity Technique #1

Approach:
1. Apply development method to cumulative closed claim counts.
2. Apply development method to reported severities.
3. Multiply projected ultimate counts x projected ultimate severity.
4. Compute IBNR as projected ultimate claims minus latest reported claims.

Assumption emphasis:
- Count and severity patterns observed to date remain representative for future development.


In [ ]:
count_dev = cl.Development(average='volume', n_periods=-1).fit_transform(closed_count_triangle)
severity_dev = cl.Development(average='volume', n_periods=-1).fit_transform(reported_severity_triangle)

count_model = cl.Chainladder().fit(count_dev)
severity_model = cl.Chainladder().fit(severity_dev)

ultimate_counts_t1 = count_model.ultimate_.to_frame().iloc[:, 0]
ultimate_severity_t1 = severity_model.ultimate_.to_frame().iloc[:, 0]
ultimate_claims_t1 = ultimate_counts_t1 * ultimate_severity_t1
ibnr_t1 = ultimate_claims_t1 - latest_claims

tech1_ay = pd.DataFrame(
    {
        'UltimateCounts_T1': ultimate_counts_t1.values,
        'UltimateSeverity_T1': ultimate_severity_t1.values,
        'UltimateClaims_T1': ultimate_claims_t1.values,
        'LatestReportedClaims': latest_claims.values,
        'IBNR_T1': ibnr_t1.values,
    },
    index=ultimate_counts_t1.index.year,
)
tech1_ay.index.name = 'AccidentYear'
tech1_ay.loc['Total'] = tech1_ay.sum()
tech1_ay


## Frequency-Severity Technique #2

Approach:
1. Start from projected ultimate counts.
2. Convert to frequency by dividing by exposure.
3. Select frequency and severity using trend analysis (especially for immature years).
4. Project ultimate claims as: Exposure x Selected Frequency x Selected Severity.

In this demo, premium from `xyz.csv` is used as an exposure proxy.


In [ ]:
exposure_proxy = xyz_df.groupby('AccidentYear')['Premium'].max().sort_index()
exposure_proxy.index = pd.to_datetime(exposure_proxy.index.astype(str) + "-01-01")
exposure_proxy = exposure_proxy.reindex(ultimate_counts_t1.index)

frequency_from_t1 = ultimate_counts_t1 / exposure_proxy
severity_from_t1 = ultimate_severity_t1.copy()

mature_mask = latest_age >= 84
ay_numeric = pd.Series(ultimate_counts_t1.index.year, index=ultimate_counts_t1.index, dtype=float)

freq_fit_mask = mature_mask & frequency_from_t1.notna() & (frequency_from_t1 > 0)
sev_fit_mask = mature_mask & severity_from_t1.notna() & (severity_from_t1 > 0)

if int(freq_fit_mask.sum()) >= 2:
    freq_slope, freq_intercept = np.polyfit(
        ay_numeric[freq_fit_mask].to_numpy(dtype=float),
        np.log(frequency_from_t1[freq_fit_mask].to_numpy(dtype=float)),
        1,
    )
    selected_frequency_t2 = np.exp(freq_intercept + freq_slope * ay_numeric.to_numpy(dtype=float))
else:
    selected_frequency_t2 = np.repeat(float(frequency_from_t1.mean()), len(ay_numeric))

if int(sev_fit_mask.sum()) >= 2:
    sev_slope, sev_intercept = np.polyfit(
        ay_numeric[sev_fit_mask].to_numpy(dtype=float),
        np.log(severity_from_t1[sev_fit_mask].to_numpy(dtype=float)),
        1,
    )
    selected_severity_t2 = np.exp(sev_intercept + sev_slope * ay_numeric.to_numpy(dtype=float))
else:
    selected_severity_t2 = np.repeat(float(severity_from_t1.mean()), len(ay_numeric))

selected_frequency_t2 = pd.Series(selected_frequency_t2, index=ultimate_counts_t1.index)
selected_severity_t2 = pd.Series(selected_severity_t2, index=ultimate_counts_t1.index)

ultimate_claims_t2 = exposure_proxy * selected_frequency_t2 * selected_severity_t2
ibnr_t2 = ultimate_claims_t2 - latest_claims

tech2_ay = pd.DataFrame(
    {
        'ExposureProxy': exposure_proxy.values,
        'SelectedFrequency_T2': selected_frequency_t2.values,
        'SelectedSeverity_T2': selected_severity_t2.values,
        'UltimateClaims_T2': ultimate_claims_t2.values,
        'LatestReportedClaims': latest_claims.values,
        'IBNR_T2': ibnr_t2.values,
    },
    index=ultimate_claims_t2.index.year,
)
tech2_ay.index.name = 'AccidentYear'
tech2_ay.loc['Total'] = tech2_ay.sum()
tech2_ay


## Frequency-Severity Method Comparison

Pros often cited:
- More stable than pure development for early maturities.
- Gives clearer process insight by separating count and severity drivers.

Cons often cited:
- Requires additional assumptions (trend, exposure mapping, severity stability).
- Data requirements are higher than single-triangle development methods.


In [ ]:
freq_sev_summary = pd.Series(
    {
        'UltimateClaims_T1_Total': float(ultimate_claims_t1.sum()),
        'IBNR_T1_Total': float(ibnr_t1.sum()),
        'UltimateClaims_T2_Total': float(ultimate_claims_t2.sum()),
        'IBNR_T2_Total': float(ibnr_t2.sum()),
        'T2_minus_T1_Ultimate': float(ultimate_claims_t2.sum() - ultimate_claims_t1.sum()),
    }
)

freq_sev_assumption_check = pd.DataFrame(
    {
        'MatureForTrend': mature_mask.values,
        'LatestMaturityAge': latest_age.values,
        'ObservedFrequencyFromT1': frequency_from_t1.values,
        'SelectedFrequency_T2': selected_frequency_t2.values,
        'ObservedSeverityFromT1': severity_from_t1.values,
        'SelectedSeverity_T2': selected_severity_t2.values,
    },
    index=ultimate_counts_t1.index.year,
)
freq_sev_assumption_check.index.name = 'AccidentYear'

freq_sev_summary, freq_sev_assumption_check


## Disposal Rate Method

Workflow:
1. Project/select ultimate closed counts.
2. Select disposal rates by maturity age.
3. Project future closed-count emergence from selected disposal rates.
4. Select incremental severities by maturity and apply to projected future closed counts.
5. Compute unpaid/IBNR.

Useful formulas used in the code:
- U_i = C_i,t / DR_t
- Projected incremental closed counts between ages a and b: U_i x (DR_b - DR_a)


In [ ]:
disp_triangle = cl.Triangle(
    disp_df,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Disposal Rate', 'Closed Claim Counts', 'Paid Claims'],
    cumulative=True,
)

dr_triangle = disp_triangle['Disposal Rate']
disp_closed_triangle = disp_triangle['Closed Claim Counts']
disp_paid_triangle = disp_triangle['Paid Claims']

disp_count_dev = cl.Development(average='volume', n_periods=-1).fit_transform(disp_closed_triangle)
disp_count_model = cl.Chainladder().fit(disp_count_dev)
ultimate_closed_counts = disp_count_model.ultimate_.to_frame().iloc[:, 0]

dr_long = triangle_to_frame(dr_triangle, origin_as_datetime=False).reset_index()
dr_matrix = dr_long.pivot(index='origin', columns='development', values='Disposal Rate').sort_index().sort_index(axis=1)
disp_closed_long = triangle_to_frame(disp_closed_triangle, origin_as_datetime=False).reset_index()
disp_closed_matrix = disp_closed_long.pivot(index='origin', columns='development', values='Closed Claim Counts').sort_index().sort_index(axis=1)
disp_paid_long = triangle_to_frame(disp_paid_triangle, origin_as_datetime=False).reset_index()
disp_paid_matrix = disp_paid_long.pivot(index='origin', columns='development', values='Paid Claims').sort_index().sort_index(axis=1)

age_cols = disp_closed_matrix.columns.astype(int)
selected_dr_by_age = pd.Series(index=age_cols, dtype=float)
for age in age_cols:
    vals = dr_matrix[age].dropna()
    selected_dr_by_age.loc[age] = vals.tail(5).mean() if len(vals) >= 5 else vals.mean()
selected_dr_by_age = selected_dr_by_age.ffill().clip(upper=1.0)
if pd.isna(selected_dr_by_age.iloc[-1]) or selected_dr_by_age.iloc[-1] < 0.999:
    selected_dr_by_age.iloc[-1] = 1.0

inc_paid_matrix = disp_paid_matrix.diff(axis=1)
inc_paid_matrix.iloc[:, 0] = disp_paid_matrix.iloc[:, 0]
inc_closed_matrix = disp_closed_matrix.diff(axis=1)
inc_closed_matrix.iloc[:, 0] = disp_closed_matrix.iloc[:, 0]
inc_severity_matrix = inc_paid_matrix / inc_closed_matrix

selected_inc_severity_by_age = pd.Series(index=age_cols, dtype=float)
for age in age_cols:
    vals = inc_severity_matrix[age].replace([np.inf, -np.inf], np.nan).dropna()
    selected_inc_severity_by_age.loc[age] = vals.tail(5).median() if len(vals) >= 5 else vals.median()
selected_inc_severity_by_age = selected_inc_severity_by_age.ffill()

tail_mask = selected_inc_severity_by_age.index >= 108
if tail_mask.any() and selected_inc_severity_by_age[tail_mask].notna().any():
    selected_tail_severity = float(selected_inc_severity_by_age[tail_mask].dropna().mean())
    selected_inc_severity_by_age.loc[tail_mask] = selected_tail_severity

selected_dr_by_age, selected_inc_severity_by_age


## Disposal Projection Results and Tail-Severity Considerations

Tail-severity selection notes:
- Late maturities often have sparse counts and unstable incremental severities.
- A common practice is to blend late-age severities into a selected tail severity.
- The impact should be judged relative to how much claim count remains open at those ages.


In [ ]:
latest_closed = disp_closed_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_paid_disp = disp_paid_triangle.latest_diagonal.to_frame().iloc[:, 0]

latest_age_disp = disp_closed_matrix.notna().iloc[:, ::-1].idxmax(axis=1)
latest_age_disp.index = pd.to_datetime(latest_age_disp.index.astype(str) + "-01-01")
latest_age_disp = latest_age_disp.reindex(ultimate_closed_counts.index)

projection_rows = []
for idx in ultimate_closed_counts.index:
    ay = idx.year
    latest_age_ay = int(latest_age_disp.loc[idx])
    u_closed = float(ultimate_closed_counts.loc[idx])
    latest_paid_ay = float(latest_paid_disp.loc[idx]) if pd.notna(latest_paid_disp.loc[idx]) else 0.0

    if latest_age_ay in selected_dr_by_age.index:
        dr_prev = float(selected_dr_by_age.loc[latest_age_ay])
    else:
        dr_prev = float(selected_dr_by_age[selected_dr_by_age.index <= latest_age_ay].iloc[-1])

    projected_future_unpaid = 0.0
    for age in age_cols:
        age = int(age)
        if age <= latest_age_ay:
            continue
        dr_age = float(selected_dr_by_age.loc[age])
        projected_incremental_closed = max(u_closed * (dr_age - dr_prev), 0.0)
        selected_sev_age = float(selected_inc_severity_by_age.loc[age])
        projected_future_unpaid += projected_incremental_closed * selected_sev_age
        dr_prev = dr_age

    projected_ultimate_paid = latest_paid_ay + projected_future_unpaid
    projection_rows.append(
        {
            'AccidentYear': ay,
            'LatestAge': latest_age_ay,
            'LatestClosedCounts': float(latest_closed.loc[idx]),
            'ProjectedUltimateClosedCounts': u_closed,
            'LatestPaidClaims': latest_paid_ay,
            'ProjectedUltimatePaidClaims': projected_ultimate_paid,
            'IBNR_DisposalMethod': projected_ultimate_paid - latest_paid_ay,
        }
    )

disposal_projection = pd.DataFrame(projection_rows).set_index('AccidentYear').sort_index()
disposal_projection.loc['Total'] = disposal_projection.sum()
disposal_projection


In [ ]:
final_summary = pd.DataFrame(
    {
        'Method': [
            'Freq-Sev Technique #1',
            'Freq-Sev Technique #2',
            'Disposal Rate Method',
        ],
        'TotalUltimateClaims': [
            float(ultimate_claims_t1.sum()),
            float(ultimate_claims_t2.sum()),
            float(disposal_projection.loc['Total', 'ProjectedUltimatePaidClaims']),
        ],
        'TotalIBNR': [
            float(ibnr_t1.sum()),
            float(ibnr_t2.sum()),
            float(disposal_projection.loc['Total', 'IBNR_DisposalMethod']),
        ],
    }
)

final_notes = pd.DataFrame(
    [
        ['Technique #1', 'Separates counts and severity development directly; sensitive to severity selection.'],
        ['Technique #2', 'Adds explicit exposure/frequency/severity trend structure; stronger assumption load.'],
        ['Disposal Rate', 'Useful when closed-count emergence is credible and incremental severity is stable enough by maturity.'],
    ],
    columns=['Method', 'Interpretation'],
)

final_summary, final_notes
